
# Notebook creativo: análisis EDA con `dataspark.eda`

Objetivo: ejemplificar el módulo EDA con un flujo práctico y algo más creativo.

Flujo:
1. Cargar datos.
2. Limpiar **rápidamente** con `dataspark.cleansing`.
3. Analizar con `DataExplorer`, `DistributionAnalyzer`, `CorrelationAnalyzer` y `PlotFactory`.



## Dataset

Usamos el dataset `tips` de seaborn (versión CSV pública):
- `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv`

Si falla la red, se usa fallback local (`load_wine`) para que el notebook sea ejecutable en entornos restringidos.


In [ ]:

import pandas as pd
import numpy as np

from dataspark.cleansing import DataCleaner
from dataspark.eda import DataExplorer, DistributionAnalyzer, CorrelationAnalyzer, PlotFactory

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

try:
    raw_df = pd.read_csv(url)
    source_name = f"tips (URL): {url}"
except Exception as exc:
    from sklearn.datasets import load_wine
    wine = load_wine(as_frame=True)
    raw_df = wine.frame
    source_name = "Fallback local: sklearn.datasets.load_wine()"
    print("No se pudo descargar dataset remoto:", exc)

print("Fuente:", source_name)
print("Shape original:", raw_df.shape)
raw_df.head()


## 1) Limpieza breve con `dataspark.cleansing` (sin extendernos)


In [ ]:

df = raw_df.copy()

# Ruido mínimo para demostrar limpieza
if "total_bill" in df.columns:
    df = df.rename(columns={"total_bill": " Total Bill "})

if "day" in df.columns:
    df.loc[df.sample(min(8, len(df)), random_state=7).index, "day"] = np.nan

cleaner = DataCleaner(missing_strategy="median", standardize_columns=True, strip_whitespace=True)
df_clean = cleaner.fit_transform(df)

print("Shape limpio:", df_clean.shape)
print("Nulos totales:", int(df_clean.isna().sum().sum()))
df_clean.head()



## 2) `DataExplorer`: perfil general

Aquí evaluamos:
- estadísticos descriptivos extendidos,
- estructura de tipos y nulos,
- señal de normalidad por variable.


In [ ]:

explorer = DataExplorer(df_clean)
summary_df = explorer.summary()
summary_df.head(10)


In [ ]:

info = explorer.info_report()
info


In [ ]:

normality = explorer.normality_tests(alpha=0.05)
normality.sort_values("p_value").head(10)



## 3) `DistributionAnalyzer`: ¿qué distribución explica mejor los montos?

Análisis creativo:
- comparamos distribuciones candidatas para una variable monetaria,
- evaluamos si hay señal de multimodalidad,
- revisamos percentiles extremos (q01, q99).


In [ ]:

dist = DistributionAnalyzer(df_clean)

target_col = "total_bill" if "total_bill" in df_clean.columns else df_clean.select_dtypes(include="number").columns[0]
print("Columna objetivo:", target_col)

fits = dist.fit_distributions(target_col)
fits.head(5)


In [ ]:

modality = dist.detect_multimodality(target_col)
quantiles = dist.quantile_analysis(target_col)

print("Multimodalidad:", modality)
print("Quantiles clave:", {k: v for k, v in quantiles["quantiles"].items() if k in ["q01", "q50", "q99"]})



## 4) `CorrelationAnalyzer`: relaciones útiles y significativas

Preguntas:
- ¿qué pares tienen mayor asociación?
- ¿cuáles son estadísticamente significativos?
- si existe variable binaria, ¿qué tan asociada está con una continua?


In [ ]:

corr_an = CorrelationAnalyzer(df_clean)

corr_matrix = corr_an.correlation_matrix(method="pearson")
corr_matrix


In [ ]:

top_corr = corr_an.top_correlations(n=8, method="pearson")
top_corr


In [ ]:

pairs_sig = corr_an.pairwise_significance(method="spearman", alpha=0.05)
pairs_sig.head(10)


In [ ]:

# Point-biserial sólo si hay columna binaria numérica
num_cols = df_clean.select_dtypes(include="number").columns.tolist()
binary_candidates = [c for c in num_cols if set(df_clean[c].dropna().unique()).issubset({0, 1})]

if binary_candidates and len(num_cols) > 1:
    binary_col = binary_candidates[0]
    continuous_col = [c for c in num_cols if c != binary_col][0]
    pb = corr_an.point_biserial(binary_col=binary_col, continuous_col=continuous_col)
    print(pb)
else:
    print("No hay variable binaria numérica clara para point-biserial en este dataset.")



## 5) `PlotFactory`: visualizaciones para contar una historia

Generamos figuras (sin mostrarlas todas en exceso) para:
- mapa de nulos,
- matriz de correlación,
- distribución de numéricas,
- boxplots para outliers.


In [ ]:

plotter = PlotFactory(style="whitegrid", figsize=(10, 6))

fig_missing = plotter.missing_heatmap(df_clean)
fig_corr = plotter.correlation_heatmap(df_clean, method="pearson")

numeric_cols = df_clean.select_dtypes(include="number").columns.tolist()
subset_cols = numeric_cols[:min(6, len(numeric_cols))]

fig_dist = plotter.distribution_grid(df_clean, columns=subset_cols)
fig_box = plotter.boxplot_grid(df_clean, columns=subset_cols)

# Guardado opcional para reporte
plotter.save(fig_missing, "artifacts_eda_missing_heatmap.png")
plotter.save(fig_corr, "artifacts_eda_corr_heatmap.png")
plotter.save(fig_dist, "artifacts_eda_distribution_grid.png")
plotter.save(fig_box, "artifacts_eda_boxplot_grid.png")

print("Figuras guardadas en el directorio actual.")



## 6) Conclusión rápida

- Limpiamos de forma breve con `DataCleaner`.
- Ejecutamos un análisis estadístico más creativo con:
  - calidad/normalidad,
  - ajuste de distribuciones,
  - correlaciones y significancia,
  - visualizaciones de apoyo.

Este patrón se puede reusar para exploración previa a modelado.
